In [1]:
import os
import rasterio
from importlib_resources import files
from pathlib import Path
from rasterio.enums import Resampling
from tqdm import tqdm

from beak.utilities.raster_processing import unify_raster_grids
from beak.utilities.io import save_raster, create_file_list, check_path, create_file_folder_list


**User inputs**
1. Select the root folder where the rasters are stored.
2. Then, go down to select the subfolders that contain the rasters to be unified.<p>

The script will search for all rasters and store them in relative paths.

In [2]:
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 2240
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "us_cont"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")
BASE_EVIDENCE = "geology"

# Export
# Some data sets are already **LOG** scaled and need special actions. They will be unified and stored in a separate folder.
PATH_INPUT = BASE_PATH / "RAW" / BASE_EVIDENCE / "SGMC" / "US_CONT"

PATH_ROOT = BASE_PATH / "PROCESSED" / str("national" + "_" + BASE_EXTENT + "_" + BASE_SPATIAL)
PATH_EXPORT = PATH_ROOT / "unified" / BASE_EVIDENCE
PATH_EXPORT_LOG = PATH_ROOT / "unified_scaled_log" / BASE_EVIDENCE

CURRENT_DIR = Path(os.getcwd()) / PATH_EXPORT.name
OUT_FOLDER = PATH_EXPORT
OUT_FOLDER_LOG = PATH_EXPORT_LOG

print(f"Input folder: {PATH_INPUT}")
print(f"Export folder: {OUT_FOLDER}")

Input folder: /beak-ta3/src/beak/data/RAW/geology/SGMC/US_CONT
Export folder: /beak-ta3/src/beak/data/PROCESSED/national_us_cont_102008_2240/unified/geology


Select subfolders to be scaled.

In [3]:
folders = os.listdir(PATH_INPUT)
for folder in folders:
  if os.path.isdir(os.path.join(PATH_INPUT, folder)):
    print(folder)


Age
Distance


In [4]:
SELECTION = ["Distance"]

input_folders = [PATH_INPUT / folder for folder in SELECTION]

print("Selected folders:")
for folder in input_folders:
  print(folder)

Selected folders:
/beak-ta3/src/beak/data/RAW/geology/SGMC/US_CONT/Distance


**Select files**

In [5]:
# Create file list
file_list = []
for folder in input_folders:
  files = create_file_list(folder, recursive=True)
  file_list.extend(files)

# Create file list for log scaled data
log_files = [file.stem for file in file_list if "Log" in file.stem]

# Show results
print(f"Found {len(file_list)} files including {len(log_files)} log scaled files.")

Found 3 files including 0 log scaled files.


**Run unification**

In [6]:
# Select to write output file
dry_run = False

if dry_run:
  print("Dry run, no files will be written.\n")

# Unify
for file in tqdm(file_list, total=len(file_list)):
    folder_relative = os.path.relpath(file.parent, PATH_INPUT)

    raster = rasterio.open(file)
    unified_raster, unified_meta = unify_raster_grids(BASE_RASTER, [file], resampling_mode="auto", same_extent=True, same_shape=True)[0]

    if not dry_run:
      if file.stem in log_files:
        log_folder = OUT_FOLDER_LOG / str(folder_relative).lower()
        log_out_path = log_folder / file.name
        check_path(Path(os.path.dirname(log_out_path)))
        save_raster(log_out_path, array=unified_raster, dtype="float32", metadata=unified_meta)
      else:
        output_folder = OUT_FOLDER / str(folder_relative).lower()
        out_path = output_folder / file.name
        check_path(Path(os.path.dirname(out_path)))
        save_raster(out_path, array=unified_raster, dtype="float32", metadata=unified_meta)


 67%|██████▋   | 2/3 [00:06<00:03,  3.46s/it]

File already exists: Geol_SGMC_Dist_Jurassic_Oligocene_Intrusive.tif.


100%|██████████| 3/3 [00:08<00:00,  2.79s/it]

File already exists: Geol_SGMC_Dist_Jurassic_Oligocene_Volcanic.tif.
